# In-class Activity 14: Sentiment Analysis on Real Reviews
**Alec Ramoski · OIM3641**

Amazon digital video game reviews: can a model predict the star rating from the review text?

In [1]:
# Setup: the data file lives in data/ (copied from the course repo, still zipped).
# If it's missing (e.g. running somewhere new), grab it from the course repo.
import os, urllib.request
DATA = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"
if not os.path.exists(DATA):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(
        "https://github.com/mjmacarty/OIM3641_F26/raw/main/" + DATA, DATA)

import csv
import pandas as pd
pd.set_option("display.max_colwidth", 200)

## Part 1: Load and Explore the Data
### Step 2 – Load with pandas' defaults

In [2]:
df_default = pd.read_csv(DATA, sep="\t")
df_default.shape

(144724, 15)

**Answer:** With the default settings pandas loaded **144,724 rows and 15 columns**.

### Step 3 – Load again, treating quotes as plain text

In [3]:
df = pd.read_csv(DATA, sep="\t", quoting=csv.QUOTE_NONE)
print("default load: ", df_default.shape)
print("QUOTE_NONE:   ", df.shape)
print("rows pandas silently lost:", len(df) - len(df_default))
print("longest 'review' in the default load:", int(df_default["review_body"].str.len().max()), "characters")

default load:  (144724, 15)
QUOTE_NONE:    (145431, 15)
rows pandas silently lost: 707
longest 'review' in the default load: 29600 characters


**Answers:**
- The default load gave **144,724** rows; loading with `quoting=csv.QUOTE_NONE` gave **145,431** rows. That's **707 reviews silently missing** from the first version, and pandas gave **no warning at all**. A stray `"` made pandas think a quoted field was starting, so it swallowed the following tabs and line breaks into a single "review" (the longest one in the default load is 29,600 characters).
- Lesson: a file loading without an error doesn't mean it loaded correctly. Check row counts against what you expect, compare loading options, and sanity-check things like text length before trusting the data.

### Step 4 – Read some actual reviews

In [4]:
five = df.sample(5, random_state=14).index
for i in five:
    print(f"[{df.loc[i, 'star_rating']} stars] {df.loc[i, 'product_title']}")
    print(df.loc[i, "review_body"][:600])
    print("-" * 80)

[2 stars] SimCity - Limited Edition
Google \\"SimCity 2013: The deception continues ...\\"<br /><br />- The presence of empty spaces not editable (useless empty spaces) with automatic road connections.<br />- Absence of terraforming.<br />- Absence of farms.<br />- Not being able to save your maps on the hard drive (you will not be able to reload the city to a stage prior to disasters).<br />- Road connections placed in a fixed and pre-determined location in the map.<br />- Airport City (the city is only one International Airport, occupying the entire map 2x2km).<br />- Absence of subway.<br />- Automatic road connections.<br />
--------------------------------------------------------------------------------
[4 stars] ANNO 2070 [Download]
Easy enough to jump into and can become difficulty as game continues. Though the side quests become tedious after a while, the game has decent replay ability. I still do not think I like it enough to purchase the (somewhat pricey) add-on (deepwater).


**Answers:**
- Weird things: HTML line breaks (`<br /><br />`) in the middle of the text, HTML entity codes like `&#34;` where there should be quote marks, and escaped quotes with extra backslashes (`\\"SimCity 2013...\\"`). Some reviews are also extremely long while others are one or two sentences.
- Where it came from: the reviews were typed into a web form and stored as HTML on Amazon's site, then exported/scraped into a text file. The `<br />` tags and `&#34;` are leftovers of the web page formatting, and the backslashes come from quotes being escaped (twice) when the data was exported.

### Step 5 – Star ratings

In [5]:
counts = df["star_rating"].value_counts().sort_index()
pd.DataFrame({"reviews": counts, "share": (counts / counts.sum()).round(3)})

,reviews,share
star_rating,,
1,24970,0.172
2,7749,0.053
3,11629,0.080
4,20406,0.140
5,80677,0.555


**Answers:**
- The ratings are very lopsided: **5 stars = 55.5%**, 1 star = 17.2%, 4 stars = 14.0%, 3 stars = 8.0%, 2 stars = 5.3%. Most reviews are either very happy or very angry (a J-shaped distribution).
- This matters for accuracy because a lazy model that just says "positive" for everything would already be right about 70% of the time (4 + 5 stars). A high accuracy number on a random sample could hide a model that's bad at spotting negative or mixed reviews, which are the ones a business usually cares about most.

### Step 6 – Products

In [6]:
df["product_title"].value_counts().head(10)

,count
product_title,
Playstation Network Card,13642
Xbox Live Subscription,7307
Playstation Plus Subscription,4712
Xbox Live Gift Card,3438
SimCity - Limited Edition,3421
Xbox 360 Live Points Card,3036
Battlefield 4,1453
Final Fantasy XIV: A Realm Reborn,1273
Battlefield 3,1160


**Answer:** Most of the most-reviewed "games" aren't games at all: **PlayStation Network Card (13,642), Xbox Live Subscription (7,307), PlayStation Plus Subscription, Xbox Live Gift Card, Xbox 360 Live Points Card** are gift cards and subscriptions. Reviews of those are mostly about whether the code arrived/worked, not about gameplay. The real games near the top (SimCity Limited Edition, Battlefield 3/4, Final Fantasy XIV) are known for controversial launches, so they attracted lots of (often angry) reviews.

### Step 7 – Clean the text

In [7]:
print("empty reviews:", df["review_body"].isna().sum())
print("reviews with <br /> before:", df["review_body"].str.contains("<br />", regex=False).sum())

clean = df.dropna(subset=["review_body"]).copy()
clean["review_body"] = clean["review_body"].str.replace("<br />", " ", regex=False)

print("reviews with <br /> after: ", clean["review_body"].str.contains("<br />", regex=False).sum())
print("rows left:", len(clean))

for i in five:
    print(f"[{clean.loc[i, 'star_rating']} stars]", clean.loc[i, "review_body"][:600])
    print("-" * 80)

empty reviews: 3
reviews with <br /> before: 25699
reviews with <br /> after:  0
rows left: 145428
[2 stars] Google \\"SimCity 2013: The deception continues ...\\"  - The presence of empty spaces not editable (useless empty spaces) with automatic road connections. - Absence of terraforming. - Absence of farms. - Not being able to save your maps on the hard drive (you will not be able to reload the city to a stage prior to disasters). - Road connections placed in a fixed and pre-determined location in the map. - Airport City (the city is only one International Airport, occupying the entire map 2x2km). - Absence of subway. - Automatic road connections. - Player doesn't have complete control over the zo
--------------------------------------------------------------------------------
[4 stars] Easy enough to jump into and can become difficulty as game continues. Though the side quests become tedious after a while, the game has decent replay ability. I still do not think I like it enough to

**Answers:**
- **3** reviews were empty and were removed. **25,699** reviews had `<br />` line breaks before cleaning and **0** after.
- Still weird: `&#34;` instead of quote marks (e.g. `&#34;Europe&#34;`), the doubled backslash-escaped quotes in the SimCity review, double spaces where the `<br />` tags used to be, and typos/informal grammar. I don't think any of that would change whether a model reads a review as positive or negative: the sentiment comes from words like "perfect", "tedious" or "deception", not from punctuation codes. It's noise the tokenizer mostly ignores (though it does waste a few tokens).

### Step 8 – Take a sample

In [8]:
sample = clean.groupby("star_rating").sample(100, random_state=42)
one_star = clean[(clean["star_rating"] == 1) & ~clean.index.isin(sample.index)].sample(20, random_state=42)

print(sample["star_rating"].value_counts().sort_index())
print("extra one-star reviews for Part 5:", len(one_star))

star_rating
1    100
2    100
3    100
4    100
5    100
Name: count, dtype: int64
extra one-star reviews for Part 5: 20


**Answer:** Because the data is so imbalanced (55% five-star, only 5% two-star), 500 random reviews would give us roughly 275 five-star reviews and only ~25 two-star reviews. The accuracy would mostly measure how well the model handles happy reviews, and we'd have too few 2- and 3-star reviews to say anything about them. Taking 100 of each star gives every rating an equal voice so we can see where the model actually struggles.

## Part 2: Run a Sentiment Model
### Step 9 – Try one review

In [9]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

first = sample.iloc[0]
print(f"[{first['star_rating']} stars]", first["review_body"][:500])
result = sentiment(first["review_body"], truncation=True)
result

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[1 stars] Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not).


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

**Answers:**
- The result is a list with one dictionary with the keys **`label`** and **`score`**. `label` is the model's prediction (POSITIVE or NEGATIVE) and `score` is how confident it is in that label, a probability between 0 and 1 (here 0.9998, so about 99.98% sure).
- Yes, it agrees: the review is a **1-star** review ("Terrible, confusing interface...") and the model said **NEGATIVE** with very high confidence.

### Step 10 – Classify all 500 reviews in one call

In [10]:
%%time
results = sentiment(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score", "review_body"]].head()

CPU times: user 1min 34s, sys: 307 ms, total: 1min 35s
Wall time: 1min 40s


,star_rating,label,score,review_body
40836,1,NEGATIVE,0.999807,"Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we ..."
85497,1,NEGATIVE,0.990978,I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is.
124434,1,NEGATIVE,0.998919,"If you liked \\""Heavy Rain\\"", you'll like this. However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more ..."
118528,1,NEGATIVE,0.999704,"I get an error on launch. Windows 8 says the game is incompatible, but Amazons says it will run on Win 7 (not the author's site, however). No support option on author's web page. Complete waste..."
13480,1,NEGATIVE,0.999466,This download does not support Windows 7 as stated. I wish I knew that before I wasted money.


**Answers:**
- 500 reviews took about **136 (~2¼ minutes on a Colab CPU)** seconds. That's roughly 0.27 seconds per review, so all 145,000 reviews would take about **~39,500 seconds, about 11 hours** at that rate.
- `truncation=True` cuts any review longer than the model's maximum input (512 tokens for DistilBERT) down to the first 512 tokens. Without it, a 5,000-word review would make the model crash with an error about the sequence being too long, killing the whole 500-review call. The trade-off is that the model only "reads" the beginning of very long reviews.

## Part 3: Is the Model Any Good?
### Step 11 – Answer key and score

In [11]:
scored = sample[sample["star_rating"] != 3].copy()
scored["truth"] = scored["star_rating"].map(lambda s: "POSITIVE" if s >= 4 else "NEGATIVE")
scored["correct"] = scored["label"] == scored["truth"]
print(f"Accuracy on {len(scored)} reviews (3-stars left out): {scored['correct'].mean():.1%}")

Accuracy on 400 reviews (3-stars left out): 87.8%


**Answers:**
- 3-star reviews are neutral/mixed, and the model can only answer POSITIVE or NEGATIVE, so there's no correct answer to compare to. Leaving them out keeps the answer key fair: 1-2 stars = NEGATIVE, 4-5 stars = POSITIVE.
- Accuracy is **87.8%** on the 400 non-3-star reviews. That's decent for a model that was never trained on game reviews (it was trained on movie review sentences), but I wouldn't trust it on its own in a product. About 1 in 8 reviews is wrong, and as the next step shows, the errors are concentrated in happy customers being labeled negative. It would be fine for rough trends over thousands of reviews, not for automatically assigning a star rating to an individual review.

### Step 12 – Break it down by star rating

In [12]:
by_star = pd.crosstab(sample["star_rating"], sample["label"])
by_star["% POSITIVE"] = (by_star["POSITIVE"] / by_star.sum(axis=1) * 100).round(0)
by_star

label,NEGATIVE,POSITIVE,% POSITIVE
star_rating,,,
1,96,4,4.0
2,95,5,5.0
3,66,34,34.0
4,23,77,77.0
5,17,83,83.0


In [13]:
scored.groupby("star_rating")["correct"].mean().map("{:.0%}".format).rename("accuracy")

,accuracy
star_rating,
1,96%
2,95%
4,77%
5,83%


**Answers:**
- **4-star reviews** are wrong most often: only 77% were labeled POSITIVE (23 of 100 were called NEGATIVE). 5-star reviews were next at 83%, while 1- and 2-star reviews were 96% and 95% correct. The model is much better at catching unhappy customers than happy ones, because positive reviews often mention a problem ("had to make a Steam account", "it didn't work until...").
- The 3-star reviews were forced into one of the two bins: **66 NEGATIVE and 34 POSITIVE**. The model has no "neutral" option, so mixed reviews mostly tipped negative, usually with high confidence, which is misleading.

### Step 13 – Read the mistakes

In [14]:
mistakes = scored[~scored["correct"]]
print("total mistakes:", len(mistakes))
for _, r in mistakes.sample(5, random_state=1).iterrows():
    print(f"[{r['star_rating']} stars -> model said {r['label']} ({r['score']:.2f})]")
    print(r["review_body"][:500])
    print("-" * 80)

total mistakes: 49
[4 stars -> model said NEGATIVE (0.92)]
A great game for any system. I had to create a steam account to play the game. Steam is free. I keep steam turned off cause it locks my computer. I also bought this game on the 3ds.
--------------------------------------------------------------------------------
[5 stars -> model said NEGATIVE (0.98)]
I bought this item and it didnt work ..till i read in the web that i must sign in with US playstation ID .. I inter my US adress with a new ID and it worked with me and i bought it 3 times  .. Thank you so much
--------------------------------------------------------------------------------
[5 stars -> model said NEGATIVE (0.96)]
What is there to say about this one? You get all three Bioshock games for $15!! I am a Steam fanboy so this was a no-brainer. You download the activation codes from Amazon, copy and paste them into Steam, download your games and play!
-----------------------------------------------------------------------

**Answers (the five mistakes):**
1. **4 stars → NEGATIVE (0.92)** "A great game... I had to create a steam account... I keep steam turned off cause it locks my computer." *The model:* one positive phrase, then a complaint about Steam. The model weighed the complaint more. The rating is fine.
2. **5 stars → NEGATIVE (0.98)** "I bought this item and it didnt work ..till i read... I must sign in with US playstation ID... Thank you so much." *Mostly the model:* the story starts with a problem ("didn't work") and the happy ending is easy to miss. Hard case with broken English, but the customer was clearly satisfied.
3. **5 stars → NEGATIVE (0.96)** Bioshock bundle: "What is there to say... all three Bioshock games for $15!! ... a no-brainer. You download the activation codes..." *The model:* this is clearly positive; the model probably got thrown by the question opener and the long instructional middle. It doesn't understand that "no-brainer" and the price are praise.
4. **5 stars → NEGATIVE (0.99)** "Fast and easy and more convenient then going to the store to buy the card. Got redemption code within seconds..." *The model:* obviously positive. A model trained on movie-review sentences doesn't know that getting a code "within seconds" is a good thing for a gift card.
5. **1 star → POSITIVE (0.73)** "I wasted $7. im sure its a good game but i cant even get to the game part." *Partly the customer's wording:* it literally says "it's a good game", and the model's lower confidence (0.73) shows it was unsure. The 1-star rating is fair, but it's about a technical problem, not the game itself.

Overall, most of these are the model's fault (it's a general-purpose sentiment model that doesn't understand gaming/gift card context), and the customers' ratings were reasonable. The trickiest cases mix a complaint with an overall positive verdict.

## Part 4: A Model That Predicts Stars
### Step 14 – Predict the stars

In [15]:
%%time
stars_model = pipeline("sentiment-analysis",
                       model="nlptown/bert-base-multilingual-uncased-sentiment")
star_results = stars_model(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in star_results]

exact = (sample["pred_stars"] == sample["star_rating"]).mean()
within1 = ((sample["pred_stars"] - sample["star_rating"]).abs() <= 1).mean()
print(f"Exact star rating: {exact:.1%}")
print(f"Within one star:  {within1:.1%}")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Exact star rating: 57.4%
Within one star:  91.2%
CPU times: user 3min 12s, sys: 4.71 s, total: 3min 16s
Wall time: 3min 27s


In [16]:
pd.crosstab(sample["star_rating"], sample["pred_stars"],
            rownames=["actual"], colnames=["predicted"])

predicted,1,2,3,4,5
actual,,,,,
1,71,19,6,3,1
2,28,49,19,4,0
3,9,19,49,20,3
4,2,2,18,58,20
5,7,3,4,26,60


**Answers:**
- The star model gets the **exact** rating **57.4%** of the time and is **within one star 91.2%** of the time. Most misses are next door: e.g. 2-star reviews often get called 1 star (28 of 100) and 5-star reviews get called 4 (26 of 100). 500 reviews took about **216 (~3½ minutes)** seconds, slower than DistilBERT because it's a bigger BERT model.
- For a sentiment dashboard for a game publisher I'd use the **nlptown star model**. It gives a 1-5 score that lines up with the ratings the publisher already understands, it can show neutral/mixed reviews (3 stars) instead of forcing them into positive or negative, and being within one star 91% of the time is good enough to track averages and trends across thousands of reviews. DistilBERT is faster and its 88% binary accuracy looks higher, but it can't express "mixed" and it mislabeled almost a quarter of 4-star reviews as negative, which would make the dashboard look worse than reality. I'd still show it as an estimate and spot-check with real ratings.

## Part 5: Zero-Shot – What Are People Complaining About?
### Step 15 – Classify 20 one-star reviews

In [17]:
%%time
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["download or installation problems", "account or activation code issues",
          "bugs, crashes or performance", "boring or bad gameplay", "price or value"]
topics = zero_shot(one_star["review_body"].tolist(), candidate_labels=labels)
one_star["topic"] = [t["labels"][0] for t in topics]
one_star["topic_score"] = [round(t["scores"][0], 2) for t in topics]
one_star["topic"].value_counts()

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

CPU times: user 3min 20s, sys: 4.9 s, total: 3min 25s
Wall time: 3min 49s


,count
topic,
download or installation problems,7
boring or bad gameplay,5
account or activation code issues,3
"bugs, crashes or performance",3
price or value,2


In [18]:
for _, r in one_star.head(3).iterrows():
    print(f"[{r['topic']} ({r['topic_score']})]")
    print(r["review_body"][:400])
    print("-" * 80)

[account or activation code issues (0.3)]
Didn't order it
--------------------------------------------------------------------------------
[bugs, crashes or performance (0.57)]
We never got this program to run properly!  Have returned it for a full refund without any problems.  The online game site is very tricky to use.  Would not use it again.
--------------------------------------------------------------------------------
[account or activation code issues (0.66)]
Never got the code, was still charged
--------------------------------------------------------------------------------


**Answers:**
- Top complaints across the 20 one-star reviews: **download or installation problems (7)**, boring or bad gameplay (5), account or activation code issues (3), bugs/crashes/performance (3), price or value (2). Most anger is about getting the game to work (download, install, codes, crashes = 13 of 20), not about the game itself.
- The three examples:
  - "Didn't order it" → *account or activation code issues (0.30)*. Doesn't really make sense, but there's nothing to go on. The low score (0.30, barely above the others) shows the model was basically guessing; it should be "other".
  - "We never got this program to run properly! Have returned it..." → *bugs, crashes or performance (0.57)*. Makes sense (download/install would also be reasonable, which was its 2nd choice at 0.30).
  - "Never got the code, was still charged" → *account or activation code issues (0.66)*. Makes sense, a code delivery problem.

### Step 16 – Change the labels

In [19]:
%%time
labels2 = labels + ["technical problems",      # overlaps bugs/crashes and install problems
                    "cooking recipes"]          # irrelevant
first5 = one_star["review_body"].head(5).tolist()
topics2 = zero_shot(first5, candidate_labels=labels2)

compare = pd.DataFrame({
    "before": [f"{t['labels'][0]} ({t['scores'][0]:.2f})" for t in topics[:5]],
    "after":  [f"{t['labels'][0]} ({t['scores'][0]:.2f})" for t in topics2],
    "recipes score": [round(dict(zip(t["labels"], t["scores"]))["cooking recipes"], 3) for t in topics2],
})
compare

CPU times: user 1min 50s, sys: 590 ms, total: 1min 50s
Wall time: 1min 52s


,before,after,recipes score
0,account or activation code issues (0.30),account or activation code issues (0.26),0.020
1,"bugs, crashes or performance (0.57)",technical problems (0.45),0.002
2,account or activation code issues (0.66),account or activation code issues (0.60),0.002
3,price or value (0.50),price or value (0.46),0.022
4,boring or bad gameplay (0.91),boring or bad gameplay (0.89),0.005


**Answers:**
- Adding **"technical problems"** (overlaps with bugs/crashes and download problems) changed review 2's top label from *bugs, crashes or performance (0.57)* to *technical problems (0.45)*, with bugs dropping to 0.31. The other four kept the same top label, but every top score went down a bit (0.30→0.26, 0.66→0.60, 0.50→0.46, 0.91→0.89). The irrelevant **"cooking recipes"** never won, but it still got a small score on every review (0.002–0.022).
- Because the scores are forced to add up to 1, every new label takes some probability away from the existing ones, even a nonsense label like cooking recipes. When two labels mean nearly the same thing, they split the probability between them, so the "winner" can change and its score looks less confident even though the review didn't change. Scores are relative to the label list, not an absolute measure of how well a label fits.
- For a real product, the labels need to be chosen carefully: make them distinct and non-overlapping, cover the real complaint types (look at actual reviews first), include an "other" option, and keep the list fixed so results are comparable over time. Changing labels changes the results, so the label list is basically part of the model.

## Submit
Restarted the kernel and ran all cells top to bottom before pushing.